# HIXL 内存管理

RegisterMem 与 DeregisterMem 是 HIXL 注册内存与解注册的核心接口。RegisterMem 将本地内存纳入 HIXL engine 的可访问范围，使远端进程可以通过单边通信直接访问；DeregisterMem 解除注册关系，释放 HIXL 内部相关资源。

理解内存注册的关键：注册是远端访问的前提条件，未注册的地址无法参与直传，只能通过中转模式传输。

本节学习大纲如下：

- 内存注册的概念
- RegisterMem 使用方法
- DeregisterMem 使用方法
- 开发注意事项
- 代码示例


## 1. 内存注册的概念

### 1.1 注册与申请内存的区别

很多人容易混淆"申请内存"和"注册内存"，但它们是完全不同的操作：

<img src="./images/memory_register.png" alt="内存注册流程图" width="80%">

- **申请内存**：向系统申请内存空间，得到一个地址。使用 `aclrtMalloc` 或 `aclrtMallocHost`。
- **注册内存**：告诉 HIXL "这段内存可以被远端访问"。使用 `RegisterMem`。

申请只是获得空间，注册才是让远端能够访问的前提。

### 1.2 注册内存的原理

HIXL 的传输操作依赖 RDMA/HCCS 等硬件通道。这些硬件通道需要预先"知道"哪些内存地址是合法可访问的。注册就是向 HIXL engine 登记一段内存的过程，它会：

- 建立 RDMA/HCCS 访问该内存的“硬件通道”
- 在建链阶段为远端访问准备必要的权限资源
- 使远端进程可以通过单边通信直接访问这段内存

### 1.3 注册时机：建链前必须完成的原因

一个常见的错误是建链后才注册内存。这会导致远端访问失败，因为建链时 HIXL 会根据已注册的内存准备访问权限，建链后注册的内存不支持远端访问。

正确的顺序：Initialize → RegisterMem → Connect。

## 2. RegisterMem 使用方法

### 2.1 接口签名与核心结构

```cpp
Status RegisterMem(const MemDesc &mem, MemType type, MemHandle &mem_handle);
```

它涉及三个核心结构，理解它们的关系是正确使用的关键：

```cpp
struct MemDesc {
  uintptr_t addr;      // 你要注册的内存起始地址
  size_t len;          // 注册长度，单位字节
  uint8_t reserved[128] = {};  // 预留字段，保持默认
};

enum MemType {
  MEM_DEVICE,  // Device 内存，用 aclrtMalloc 申请
  MEM_HOST     // Host 内存，用 aclrtMallocHost 申请
};

using MemHandle = void *;  // 注册成功后返回的句柄
```

这三个结构之间的关系：你用 `MemDesc` 描述一段内存，用 `MemType` 说明内存类型，成功后得到 `MemHandle` 用于后续管理。


### 2.2 MemDesc 参数

`MemDesc` 的 `addr` 和 `len` 定义了一段连续内存的边界，后续传输只能使用这个区间内的子区间，超出边界会报错。

一个常见做法是注册一个较大的 buffer（比如 4MB），然后在传输时切分子区间使用。这样可以减少注册次数，避免频繁注册/解注册带来的开销。

### 2.3 MemType 参数

`MemType` 必须与内存申请方式匹配：

| MemType | 申请方式 | 适用场景 |
| --- | --- | --- |
| `MEM_DEVICE` | `aclrtMalloc(..., ACL_MEM_MALLOC_HUGE_ONLY)` | NPU 之间的 D2D 直传 |
| `MEM_HOST` | `aclrtMallocHost` | Host 参与的通信场景 |

**申请方式和 MemType 不匹配的后果**：用 `aclrtMalloc` 申请 Device 内存却按 `MEM_HOST` 注册，会导致建链或传输阶段失败。

开发时要确保申请方式和注册类型一致：

- Device 内存 → `MEM_DEVICE`
- Host 内存 → `MEM_HOST`

### 2.4 MemHandle 参数

`MemHandle` 是注册成功后返回的句柄，它的用途是传给 `DeregisterMem` 进行解注册。

**MemHandle 与 remote_addr 的区别**：

- **MemHandle**：本地 `RegisterMem` 返回的句柄，用于本地 `DeregisterMem`
- **remote_addr**：远端分配并注册的地址数值，填入 `TransferOpDesc.remote_addr` 用于远端访问

常见的错误是把本地的 `MemHandle` 发送给远端当作访问地址，`MemHandle` 是 HIXL 内部句柄，不是内存地址，远端需要的是你注册内存的真实地址数值（`MemDesc.addr`）。


### 2.5 返回值处理

| 返回值 | 含义 | 处理建议 |
| --- | --- | --- |
| `SUCCESS` | 内存已纳入可访问范围 | 进入建链阶段 |
| `PARAM_INVALID` | 参数错误 | 检查 addr/len/type 是否匹配申请方式 |
| 其他 | 前置条件不满足 | 打印返回码和 `aclGetRecentErrMsg()` |

RegisterMem 失败时，不要继续调用 Connect，需要先检查参数是否正确。


## 3. DeregisterMem 使用方法

### 3.1 接口与参数

```cpp
Status DeregisterMem(MemHandle mem_handle);
```

`mem_handle` 来自 RegisterMem 的输出，用于标识要解注册的内存段。

### 3.2 释放时机与顺序

DeregisterMem 必须在 Disconnect 之后调用。按照依赖关系，链路依赖内存注册，所以断链后才能解注册：

`Transfer 完成 → Disconnect → DeregisterMem → aclrtFree`

如果先 aclrtFree 再 DeregisterMem，这时就会出现底层地址已无效，解注册时访问非法地址报错的问题。

### 3.3 返回值处理

| 返回值 | 含义 |
| --- | --- |
| `SUCCESS` | 解注册成功 |
| `PARAM_INVALID` | mem_handle 无效 |
| 其他 | 失败 |


## 4. 开发注意事项

### 4.1 MemType 与申请方式不匹配

用 `aclrtMalloc` 申请的 Device 内存必须按 `MEM_DEVICE` 注册，用 `aclrtMallocHost` 申请的 Host 内存必须按 `MEM_HOST` 注册。不匹配会导致建链或传输失败，错误码通常是 `PARAM_INVALID` 或硬件相关错误。

### 4.2 建链后才注册

建链时 HIXL 会根据已注册的内存准备远端访问权限。建链后注册的内存不支持远端访问，远端尝试访问会报错，务必在 Connect 前完成所有内存注册。

### 4.3 注册数量上限

建议单个 HIXL 实例注册内存个数不超过 4k 个。需要注意的是注册个数越多建链耗时越长，过多易出现建链超时问题。

### 4.4 多线程 context 管理

RegisterMem/DeregisterMem 要求在 Initialize 同线程调用，或通过：

- `aclrtGetCurrentContext` 获取当前 context
- `aclrtSetCurrentContext` 切换到 Initialize 所在 context

跨线程调用会导致 context 不匹配，接口返回错误。


## 5. 代码示例

下面代码示例在 [2.2 HIXL 资源管理](./02.02_hixl_resource_management.ipynb) 的基础上补充内存注册流程：先申请内存、再注册、最后在传输完成后解注册并释放内存。RegisterMem 必须在建链前完成，DeregisterMem 必须在断链后、释放内存前完成。后续章节将在此基础上逐步补充完整的通信流程。

```c++
//engine 完成 Initialize流程
void *device_buffer = nullptr;
aclError acl_ret = aclrtMalloc(&device_buffer, kBytes, ACL_MEM_MALLOC_HUGE_ONLY);
if (acl_ret != ACL_SUCCESS) {
    printf("[ERROR] aclrtMalloc failed, ret = %d\n", acl_ret);
    return -1;
}

MemDesc desc{};
desc.addr = reinterpret_cast<uintptr_t>(device_buffer);
desc.len = kBytes;

MemHandle handle = nullptr;
Status ret = engine.RegisterMem(desc, MEM_DEVICE, handle);
if (ret != SUCCESS) {
    printf("[ERROR] RegisterMem failed, ret = %u\n", ret);
    aclrtFree(device_buffer);
    return -1;
}

// Connect 和 Transfer 将在后续章节介绍

ret = engine.DeregisterMem(handle);
if (ret != SUCCESS) {
    printf("[ERROR] DeregisterMem failed, ret = %u\n", ret);
}

acl_ret = aclrtFree(device_buffer);
if (acl_ret != ACL_SUCCESS) {
    printf("[ERROR] aclrtFree failed, ret = %d\n", acl_ret);
}
//执行 Finalize 流程
```

## 课后练习

本节介绍了内存注册的概念、RegisterMem/DeregisterMem 的使用方法和开发注意事项，请根据学习内容完成以下题目进行自测。

1. （判断题）`aclrtMalloc` 分配的内存自动被 HIXL 注册，无需调用 `RegisterMem`。

2. （判断题）建链 `Connect` 前需要完成所有本地内存的注册。

3. （单选题）以下哪种内存申请方式应使用 `MEM_HOST` 类型注册？
   A. `aclrtMalloc(..., ACL_MEM_MALLOC_HUGE_ONLY)`
   B. `aclrtMallocHost`
   C. `malloc`
   D. `new`

4. （单选题）调用 `DeregisterMem` 前应完成哪个步骤？
   A. `Finalize`
   B. `Connect`
   C. `Disconnect`
   D. `aclrtFree`

5. （多选题）`RegisterMem` 的约束包括？
   A. 建链前完成注册
   B. 建议单个 HIXL 实例注册上限 4k 个
   C. 与 `Initialize` 同线程或通过 context 切换
   D. 注册后立即可以传输

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/02.03_answer.txt